# Relational databases and first queries

After this block you can explain what tables, rows, columns, data types and keys are, open a SQLite database from Python, create tables with constraints and load data into them, and write queries that filter, sort and limit rows, including rows with missing values. The block serves the first two learning outcomes of Session 3: *explain tables, keys and relationships in a relational database* and *create tables in SQLite, load data and write SQL queries that filter and sort*.

The running example is the Berlin snapshot of Inside Airbnb (26 June 2026, CC BY 4.0) in the course database `case-study/data/airbnb/airbnb.sqlite`, with three tables: `listings` (one row per listing), `calendar` (one row per listing and night of the coming year) and `reviews` (number of reviews per listing and month). No prior knowledge of databases is assumed. The course uses **SQLite** only: the whole database is one file and Python includes it as the module `sqlite3`. Where SQLite differs from server databases such as PostgreSQL, the text says so.

*Author: course team · Licence: CC BY 4.0, except the sections marked "Adapted from", which keep their source licence (named there).*

## Overview

| Section | What it covers |
|---|---|
| Setup | building and opening the course database, the helper `q` |
| What a relational database is: tables, rows, columns and data types | tables, rows and columns, databases against CSV files, SQLite's storage classes, dates as text and true/false as 1/0, flexible typing and `STRICT` tables, `CAST` |
| Primary keys, foreign keys and relationships | primary and foreign keys, one-to-many relationships, normalisation, joins on paper, checking the keys of the course data, star schemas |
| SQLite: a database in one file | SQLite against server databases, three ways to open the file, `sqlite_master` and `PRAGMA table_info`, read-only connections |
| Creating tables and loading data: CREATE TABLE, INSERT, COPY | constraints, `INSERT`, `UPDATE` and `DELETE`, `executemany` and transactions, `COPY` and what SQLite offers instead, a strict schema on the real listings |
| Queries: SELECT, WHERE, ORDER BY, LIMIT | the clause template and its logical order, `SELECT` with aliases and `DISTINCT`, `WHERE` with `AND`/`OR`, `IN`, `BETWEEN`, `LIKE` and dates, `ORDER BY`, `LIMIT` and `OFFSET`, `CASE` |
| NULL values | three-valued logic, `IS NULL`, `COUNT(*)` against `COUNT(col)`, `COALESCE` and the sort order, whether missing prices are random |
| Worked example | a weekend flat for a family, checked against its calendar |
| Check your understanding, Exercises, Debrief | five questions, ten exercises in pairs, what to compare afterwards |

Where SQL sits in an analysis: a question becomes a query, the database returns a small result, and Python takes it from there.

```mermaid
flowchart LR
    Q["Business question"] --> S["SQL query"]
    S --> DB[("SQLite file<br/>listings, calendar,<br/>reviews")]
    DB --> R["Small result table"]
    R --> P["pandas<br/>in Python"]
    P --> A["Chart, model, report"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Setup: open the course database

The next cell builds the course database once if it is missing (a few seconds; it needs the Airbnb files in `case-study/data/airbnb/`, downloaded as `airbnb-v1.zip` from the course drive, see [case-study/README.md](../../../case-study/README.md)), opens a **connection** to it with `sqlite3.connect(path)` and defines a helper `q(sql)`. `q` runs a query with `pd.read_sql_query(sql, con)` and returns the result as a pandas DataFrame, which is easier to read than a list of tuples. The section on SQLite as a database in one file explains the connection, and block 2 explains `read_sql_query` and the `params` argument.

In [2]:
import sqlite3
import subprocess
import sys

import pandas as pd

DB = Path("case-study/data/airbnb/airbnb.sqlite")
if not DB.exists():                     # build the course database once from the Parquet files
    subprocess.run([sys.executable, "sessions/03-sql/workbooks/make_airbnb_sqlite.py"], check=True)

con = sqlite3.connect(DB)               # a connection to the database file


def q(sql, params=()):
    """Run an SQL query on the course database and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, con, params=params)

## What a relational database is: tables, rows, columns and data types

### Tables, rows and columns

A **relational database** stores data in **tables**. Each **row** is one record, for example one listing; each **column** has a name and a **data type** (whole number, decimal number, text). The set of table definitions is the **schema**. The idea goes back to Edgar F. Codd (1970), who proposed that data should be described by tables and queried by their content, not by the way they are stored on disk. A program that stores such tables and answers queries on them is a **database management system** (DBMS); SQLite, PostgreSQL, MySQL and Microsoft SQL Server are examples.

You will meet several words for the same things. A table is also called a **relation**, a row a **tuple** or **record**, and a column an **attribute** or **field**. A table looks much like a pandas DataFrame, with two differences: its rows have no built-in order, and its columns have declared types and rules that the database checks on every write.

Why not just keep CSV files? Files work well for small data that one person analyses. A database adds reliable storage that survives crashes, fast access to data that does not fit into memory, rules that block invalid values (a negative price, a review for a listing that does not exist), control over who may read and write, and safe concurrent use by several programs.

A small example that you can follow by hand (the ids are invented; the real ones have up to 19 digits):

| listings |  |  |  |
|---|---|---|---|
| **id** | district | room_type | price |
| 1 | Mitte | Entire home/apt | 120 |
| 2 | Neukölln | Private room | 45 |
| 3 | Mitte | Entire home/apt | NULL |

| reviews |  |  |
|---|---|---|
| **listing_id** | **month** | n_reviews |
| 1 | 2026-04-01 | 3 |
| 1 | 2026-05-01 | 2 |
| 2 | 2026-05-01 | 1 |

`NULL` marks a value that is missing or unknown: listing 3 shows no price. The section on NULL values treats it in full.

*Terminology and the comparison with CSV files adapted from [Data 100, SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

Let us look at the real `listings` table. `SELECT columns FROM table LIMIT 3` returns the named columns of the first three rows (the section on queries explains the clauses). We leave out the listing title on purpose: hosts sometimes write their own name into it, and an analysis does not need it.

In [3]:
q("SELECT id, host_id, district, room_type, accommodates, price, minimum_nights, first_review FROM listings LIMIT 3")

,id,host_id,district,room_type,accommodates,price,minimum_nights,first_review
0,3176,3718,Pankow,Entire home/apt,2,160.71,2,2009-06-20
1,9991,33852,Pankow,Entire home/apt,7,193.33,6,2015-08-09
2,14325,55531,Pankow,Entire home/apt,1,NaN,180,2010-06-29


### Data types in SQLite

SQLite stores every value in one of five **storage classes**:

| Storage class | What it holds | Example in the course data |
|---|---|---|
| `NULL` | a missing value | a listing without a price |
| `INTEGER` | a whole number | `id`, `accommodates`, `available` (1/0) |
| `REAL` | a decimal number (8-byte floating point) | `price`, `latitude` |
| `TEXT` | a character string | `district`, dates such as `'2026-07-01'` |
| `BLOB` | raw bytes, stored exactly as given | (not used here; images or files) |

SQLite has no separate type for dates or for true/false:

- **dates** are stored as text in the ISO form `'YYYY-MM-DD'`. Text in this form sorts in date order, and SQLite's date functions (`date()`, `strftime()`, `julianday()`) read it;
- **true/false** is stored as the integers 1 and 0. A comparison such as `price > 100` also returns 1 or 0, which the section on NULL values and block 2 use for counting.

PostgreSQL has separate `DATE`, `TIMESTAMP`, `BOOLEAN`, `NUMERIC` and many more types, and always checks them.

**Predict.** What does `typeof()` return for `id`, `price`, `first_review` and a comparison `price > 100` in the first listing?

In [4]:
q("""
SELECT typeof(id), typeof(price), typeof(first_review), typeof(price > 100), price > 100 AS is_expensive
FROM listings
LIMIT 1
""")

,typeof(id),typeof(price),typeof(first_review),typeof(price > 100),is_expensive
0,integer,real,text,integer,1


`first_review` is a date, but SQLite stores it as `text`. The comparison is an `integer`, 1 or 0. Because text dates sort correctly, `MIN` and `MAX` work on them directly. `julianday()` turns a text date into a day number, so that the difference of two dates is a number of days.

In [5]:
q("""
SELECT MIN(first_review) AS earliest_first_review,
       MAX(last_review)  AS latest_last_review,
       ROUND(julianday('2026-06-26') - julianday(MIN(first_review)) ) AS days_since_earliest
FROM listings
""")

,earliest_first_review,latest_last_review,days_since_earliest
0,2009-06-20,2026-07-02,6215.0


### Flexible typing and STRICT tables

A column declared `INTEGER` in SQLite *prefers* integers but, unlike most databases, also accepts other values: SQLite calls this **type affinity** or flexible typing. Text that looks like a number is converted, and text that does not is stored as text. A table declared `STRICT` (SQLite 3.37 or newer) rejects such values, as PostgreSQL would. The next cell creates both kinds of table in a temporary in-memory database (`":memory:"`, nothing is written to disk) and inserts the same three values. `try`/`except` catches the error so that the notebook keeps running.

In [6]:
mem = sqlite3.connect(":memory:")                  # a temporary database that disappears when closed
mem.execute("CREATE TABLE loose (nights INTEGER)")
mem.execute("CREATE TABLE strict_nights (nights INTEGER) STRICT")
for value in [3, "4", "three"]:
    mem.execute("INSERT INTO loose VALUES (?)", (value,))
    try:
        mem.execute("INSERT INTO strict_nights VALUES (?)", (value,))
    except sqlite3.Error as error:
        print(f"STRICT table rejects {value!r}: {error}")
print(mem.execute("SELECT nights, typeof(nights) FROM loose").fetchall())

STRICT table rejects 'three': cannot store TEXT value in INTEGER column strict_nights.nights
[(3, 'integer'), (4, 'integer'), ('three', 'text')]


The loose table converted `'4'` into the integer 4 but kept `'three'` as text: a later `AVG(nights)` would silently treat it as 0. The `STRICT` table accepted 3 and `'4'` (which converts losslessly) and rejected `'three'`.

### Converting types with `CAST`

`CAST(expression AS type)` converts a value to another type, for example a number stored as text to an integer, and `AS name` gives the result column a readable name. Without `AS`, the column is named after the whole expression.

In [7]:
q("""
SELECT price,
       CAST(price AS INTEGER) AS price_whole_euros,      -- cuts off the decimals
       CAST('12' AS INTEGER) + 1 AS text_to_number,
       CAST(accommodates AS TEXT) || ' guests' AS guests_text   -- || joins text
FROM listings
WHERE price IS NOT NULL
LIMIT 3
""")

,price,price_whole_euros,text_to_number,guests_text
0,160.71,160,13,2 guests
1,193.33,193,13,7 guests
2,10.88,10,13,2 guests


*`CAST` and column aliases adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause), here on the Airbnb data.*

### Relational tables in practice

- **Stack Exchange Data Explorer** lets anyone query the public Stack Overflow database with SQL; questions, answers, users and votes are separate tables.
- **Inside Airbnb** publishes each city as separate files (listings, calendar, reviews); the course database turns them into three tables.
- A table with named, typed columns is a contract: every row has the same fields, and every program that reads the table can rely on that. One cell of text in a spreadsheet column of prices is enough to break a calculation.

> 📖 **Read more:** [extras/01-data100-sql-basics.ipynb](extras/01-data100-sql-basics.ipynb) (UC Berkeley Data 100, BSD 3-Clause): databases against CSV files, schema and terminology on small toy tables.

## Primary keys, foreign keys and relationships

### Primary keys and foreign keys

A **primary key** is a column, or a combination of columns, whose value identifies each row uniquely and is never empty. In the case study, `id` identifies a listing. The review table has a **composite key**: the pair (`listing_id`, `month`) identifies a row, because each listing has at most one row per month. The calendar has the key (`listing_id`, `date`).

A **foreign key** is a column that refers to the primary key of another table. `calendar.listing_id` and `reviews.listing_id` point to `listings.id`. This expresses a **relationship**: one listing has many calendar nights and many review months, each of which belongs to exactly one listing (a *one-to-many* relationship). Other relationships are *one-to-one* (a listing and its single registration record) and *many-to-many* (listings and amenities: a listing has many amenities, and an amenity belongs to many listings; this needs a third table with one row per pair). The database can enforce a foreign key: a review row for a listing that does not exist is rejected.

```mermaid
classDiagram
    direction LR
    class listings {
        id : INTEGER  PK
        host_id : INTEGER
        district : TEXT
        room_type : TEXT
        price : REAL
        minimum_nights : INTEGER
        license_status : TEXT
    }
    class calendar {
        listing_id : INTEGER  FK
        date : TEXT
        available : INTEGER 0/1
    }
    class reviews {
        listing_id : INTEGER  FK
        month : TEXT
        n_reviews : INTEGER
    }
    listings "1" --> "0..*" calendar : has nights
    listings "1" --> "0..*" reviews : has review months
```

### Normalisation

Storing each fact once, in one table, is called **normalisation**. The district of a listing is stored once in `listings`, not repeated in each of its 365 calendar rows. If a listing moves to another district, one row changes, and no copy can contradict another.

### Following a relationship: joins

To answer "how many reviews did the listings in Mitte receive?", each review row needs the district of its listing: the database pairs the rows of two tables whose keys match. This operation is a **join**; block 2 teaches its syntax. Three kinds matter, shown on the small tables from the start of the notebook:

- an **inner join** returns one row for each pair of matching rows: listing 1 with its two review months and listing 2 with its one, 3 rows; listing 3 has no partner and disappears;
- a **left join** returns the same pairs and, in addition, every row of the left table without a partner, with NULL in the columns of the right table: 4 rows;
- a **cross join** pairs every row of one table with every row of the other, whether the keys match or not: 3 × 3 = 9 rows.

### Why keys matter

Keys make the relationship between tables explicit, and the database uses them to guard data quality at the moment data are written: a review row for an unknown listing, or the same night of a listing twice, is rejected immediately instead of being discovered months later in an analysis. The database also builds an **index** (a sorted lookup structure, like the index at the back of a book) on every primary key, which makes lookups and joins by key fast.

### Checking the keys of the course data

A primary key must be unique. `COUNT(*)` counts rows and `COUNT(DISTINCT column)` counts the different values of a column; if both are equal, no value occurs twice. For the composite key of the calendar, the inner query in parentheses (a **subquery**, block 2) first keeps the distinct pairs, and the outer query counts them.

**Predict.** The calendar covers 365 nights per listing. Roughly how many rows does it have?

In [8]:
q("""
SELECT (SELECT COUNT(*) FROM listings)                                         AS n_listings,
       (SELECT COUNT(DISTINCT id) FROM listings)                               AS n_ids,
       (SELECT COUNT(*) FROM calendar)                                         AS n_calendar_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, date FROM calendar)) AS n_calendar_keys,
       (SELECT COUNT(*) FROM reviews)                                          AS n_review_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, month FROM reviews)) AS n_review_keys
""").T.rename(columns={0: "value"})

,value
n_listings,10221
n_ids,10221
n_calendar_rows,3759500
n_calendar_keys,3759500
n_review_rows,180656
n_review_keys,180656


All three keys hold: each count of rows equals the count of distinct keys. Now the foreign keys: does every `listing_id` in `calendar` and `reviews` refer to an existing listing? `WHERE listing_id NOT IN (SELECT id FROM listings)` keeps the rows whose listing is missing. (This form is safe here because `id` is never NULL; block 2 shows why `NOT IN` is dangerous when the list can contain a NULL.)

In [9]:
q("""
SELECT 'calendar' AS child_table, COUNT(*) AS orphan_rows, COUNT(DISTINCT listing_id) AS unknown_listings
FROM calendar WHERE listing_id NOT IN (SELECT id FROM listings)
UNION ALL
SELECT 'reviews', COUNT(*), COUNT(DISTINCT listing_id)
FROM reviews WHERE listing_id NOT IN (SELECT id FROM listings)
""")

,child_table,orphan_rows,unknown_listings
0,calendar,28835,79
1,reviews,499,57


(`UNION ALL` stacks the rows of two queries with the same columns.) The foreign key fails for 79 listings in the calendar (28,835 rows, exactly 365 per listing) and 57 listings in the review table. Inside Airbnb scrapes listings, calendars and reviews separately, and the listings were scraped between 26 June and 3 July 2026: a listing that went offline between two scrapes appears in one file but not in the other. A real foreign key would have rejected these rows, which is exactly the point: the database makes you decide what to do with them instead of letting a join lose them silently. The course database keeps all rows, so that you can find them with SQL in block 2.

The scraping over several days also shows in the dates of the calendar:

In [10]:
q("SELECT MIN(date) AS first_night, MAX(date) AS last_night, COUNT(DISTINCT date) AS n_dates FROM calendar")

,first_night,last_night,n_dates
0,2026-06-26,2027-07-02,372


The calendar has 372 different dates, although each listing has exactly 365 rows: each calendar starts on the day its listing was scraped. Questions about "the next weeks" therefore mean slightly different weeks for different listings.

> **Note.** A file is not a database. A Parquet or CSV file has no keys and no constraints: nothing stops a duplicated `id` or a calendar row for a listing that is not in the listings file. When you work with files, check the key properties yourself (as above) and repeat the check after each update.

### Star schemas

Analytical databases often organise their tables as a **star schema**. A central **fact table** holds one row per event or measurement (a sale, a booked night, a review month) and refers by foreign keys to several **dimension tables** that describe the parts of the fact in detail (the product, the customer, the date). The Data 100 example is a shop that sells bubble tea: the fact table lists each offering by keys, and dimension tables describe tea varieties and toppings.

![A fact table with dimension tables: tea, toppings and the offerings that combine them](extras/images/multidimensional.png)

Drawing the foreign keys as lines gives the star its name:

![The same tables as a star schema, with the fact table in the centre](extras/images/star.png)

The course database is a small star: `listings` is the dimension that describes each listing, and `calendar` and `reviews` are fact tables whose rows refer to it.

*Figures and the star-schema example from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### Keys and star schemas in practice

- **Online shops** keep customers, orders and products in separate tables; an order row refers to a customer id and a product id, so that a change of address is made once.
- **Data quality checks** in finance and retail look for records that refer to a deleted customer or a discontinued product, exactly like the calendar rows of vanished listings here.

## SQLite: a database in one file

### What SQLite is

Most database systems run as a **server**: a separate program that listens on a network port, and clients connect to it with a host name, user and password. **SQLite** is different: it is a library that runs inside your program, and the whole database (all tables, keys and indexes) is one ordinary file. There is nothing to install or start; Python includes it as the module `sqlite3`.

| | SQLite | Server databases (PostgreSQL, MySQL, ...) |
|---|---|---|
| Where the data live | one file on your disk | managed by the server |
| Setup | none | install and run a server, create users |
| Many users writing at once | one writer at a time | yes |
| Typical use | apps, phones, browsers, analysis on one machine, small teams | shared operational data of an organisation |

### Three ways to open the file

You can open the file in three ways:

1. **From Python** with `sqlite3.connect("file.sqlite")`, which returns a **connection**; this course mostly uses this way;
2. in a graphical **database browser**, such as [DB Browser for SQLite](https://sqlitebrowser.org/) (free; open the file, then type SQL in the tab "Execute SQL");
3. with the command-line program `sqlite3`, which is preinstalled on macOS and can be downloaded from [sqlite.org](https://www.sqlite.org/download.html) for other systems. Commands that start with a dot, such as `.tables` and `.schema listings`, belong to this program, not to SQL.

The SQL you learn here is standard SQL and runs, with few changes, on PostgreSQL and other server databases.

### Why a database in one file matters

A database you can copy like any file is ideal for learning and for a team project: everyone gets the same data, the queries are the same as on a server, and nothing breaks because a server is not running. The limits matter only when many people write at the same time; then an organisation uses a server database, and the SQL stays the same.

### Looking inside the file

Every SQLite file has a built-in table `sqlite_master` (also called `sqlite_schema`) that lists its tables and indexes, together with the `CREATE` statement that made each one. `sqlite3.sqlite_version` is the version of the SQLite library that Python uses. `con.execute(sql)` runs one statement, and `.fetchall()` returns the result rows as a list of tuples.

**Predict.** How many objects does `sqlite_master` list for the course file: only the three tables, or the three tables plus an index for the primary key of `listings`?

In [11]:
print("SQLite version", sqlite3.sqlite_version)
print("file size:", round(DB.stat().st_size / 1e6), "MB")
print(con.execute("SELECT type, name FROM sqlite_master").fetchall())

SQLite version 3.53.4
file size: 107 MB
[('table', 'listings'), ('table', 'calendar'), ('table', 'reviews')]


`PRAGMA` statements are SQLite's commands for information and settings. `PRAGMA table_info(listings)` lists the columns of one table with their declared type, whether they may be NULL (`notnull`) and whether they belong to the primary key (`pk`).

In [12]:
q("PRAGMA table_info(listings)")[["name", "type", "notnull", "pk"]]

,name,type,notnull,pk
0,id,INTEGER,0,1
1,name,TEXT,0,0
2,host_id,INTEGER,1,0
3,district,TEXT,1,0
4,neighbourhood,TEXT,0,0
5,latitude,REAL,0,0
6,longitude,REAL,0,0
7,room_type,TEXT,1,0
8,accommodates,INTEGER,0,0
9,bedrooms,INTEGER,0,0


The column `pk` marks `id` as the primary key of `listings`. The `calendar` and `reviews` tables of the course database were written with pandas and have no declared keys; extra 02 builds a strict version with all keys and checks.

### Read-only connections

When a program only reads, open the file read-only, so that a mistake cannot change it. The connection string then is a **URI**: `file:path?mode=ro`, with `uri=True`.

In [13]:
ro = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
try:
    ro.execute("CREATE TABLE should_fail (x INTEGER)")
except sqlite3.OperationalError as error:
    print("read-only connection refuses to write:", error)
ro.close()

read-only connection refuses to write: attempt to write a readonly database


### SQLite in practice

- According to its developers, SQLite is probably the most widely deployed database engine: it is built into every Android and iOS phone, into Firefox, Chrome and Safari, and into many desktop programs, which keep their settings and history in SQLite files ([sqlite.org/mostdeployed.html](https://www.sqlite.org/mostdeployed.html)).
- The **US Library of Congress** lists SQLite files among its recommended formats for preserving datasets.
- **Data teams** ship small reference datasets as one SQLite file, so that colleagues can query them without access to a server.

> **Note.** A database browser is useful for a first look at the tables and for trying a query. For work that must be repeated, write the query into a notebook or a script, as in this course.

## Creating tables and loading data: CREATE TABLE, INSERT, COPY

### `CREATE TABLE`, constraints and `INSERT`

`CREATE TABLE` defines a table: its name, its columns and its rules. Each column is written as `name type constraints`:

- the **name** of the column;
- the **type**, for example `INTEGER`, `REAL`, `TEXT`, `BLOB`;
- optional **constraints**, the rules every row must satisfy:
  - `PRIMARY KEY` declares the key (for a composite key, write `PRIMARY KEY (a, b)` after the columns);
  - `NOT NULL` forbids missing values;
  - `UNIQUE` forbids duplicates without making the column the key;
  - `CHECK (condition)` adds a rule for the values, for example `CHECK (price > 0)`;
  - `DEFAULT value` fills the column when an insert does not name it;
  - `REFERENCES other_table (column)` declares a foreign key.

`CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, and `DROP TABLE name` deletes a table with all its rows.

`INSERT INTO table VALUES (...)` adds rows. A statement that violates a constraint fails with an error message, and the row is not stored.

*The column format `name type constraints` and the list of constraints follow [Data 100, SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

> **Note.** For historical reasons SQLite checks foreign keys only when the connection has switched them on with `PRAGMA foreign_keys = ON`. Do this right after connecting whenever you rely on foreign keys.

### Why constraints matter

Constraints turn silent data problems into loud errors at the moment data are written. A rule written once in the schema protects every program that writes to the table, including the scripts of your teammates.

### Creating two small tables with constraints

The next cell creates the two small tables from the start of the notebook in a fresh in-memory database and fills them. `executescript` runs several statements, separated by semicolons, at once. `--` starts a comment in SQL.

In [14]:
demo = sqlite3.connect(":memory:")
demo.execute("PRAGMA foreign_keys = ON")
demo.executescript("""
CREATE TABLE listings (
    id        INTEGER PRIMARY KEY,                 -- one row per listing
    district  TEXT NOT NULL,
    room_type TEXT NOT NULL CHECK (room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'Shared room')),
    price     REAL CHECK (price > 0)               -- NULL allowed: no price shown
);
CREATE TABLE reviews (
    listing_id INTEGER NOT NULL REFERENCES listings (id),   -- foreign key
    month      TEXT NOT NULL,
    n_reviews  INTEGER NOT NULL DEFAULT 1 CHECK (n_reviews > 0),
    PRIMARY KEY (listing_id, month)                          -- composite key
);
INSERT INTO listings VALUES (1, 'Mitte', 'Entire home/apt', 120), (2, 'Neukölln', 'Private room', 45),
                            (3, 'Mitte', 'Entire home/apt', NULL);
INSERT INTO reviews VALUES (1, '2026-04-01', 3), (1, '2026-05-01', 2), (2, '2026-05-01', 1);
""")
pd.read_sql_query("SELECT * FROM listings", demo)

,id,district,room_type,price
0,1,Mitte,Entire home/apt,120.0
1,2,Neukölln,Private room,45.0
2,3,Mitte,Entire home/apt,NaN


**Predict.** Which of these four inserts does the database accept, and which rule rejects each of the others?

In [15]:
invalid = {
    "unknown listing": "INSERT INTO reviews VALUES (9, '2026-05-01', 1)",
    "same month twice": "INSERT INTO reviews VALUES (1, '2026-05-01', 4)",
    "unknown room type": "INSERT INTO listings VALUES (4, 'Mitte', 'Castle', 300)",
    "district missing": "INSERT INTO listings (id, room_type) VALUES (5, 'Private room')",
}
for label, statement in invalid.items():
    try:
        demo.execute(statement)
        print(f"{label:18} accepted")
    except sqlite3.IntegrityError as error:
        print(f"{label:18} rejected: {error}")

unknown listing    rejected: FOREIGN KEY constraint failed
same month twice   rejected: UNIQUE constraint failed: reviews.listing_id, reviews.month
unknown room type  rejected: CHECK constraint failed: room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'Shared room')
district missing   rejected: NOT NULL constraint failed: listings.district


The database rejects each invalid row, and the message names the kind of rule that failed: `FOREIGN KEY`, `UNIQUE` (the composite primary key), `CHECK` and `NOT NULL`.

### More ways to change rows: `INSERT` variants, `UPDATE` and `DELETE`

- **With a column list**, `INSERT INTO reviews (listing_id, month) VALUES (2, '2026-06-01')`, the columns not named get their `DEFAULT` (here `n_reviews = 1`) or NULL. Naming the columns also keeps the statement correct if the table later gets a new column.
- **From a query**, `INSERT INTO table SELECT ...`, copies rows from another table.
- `UPDATE table SET column = value WHERE condition` changes existing rows, and `DELETE FROM table WHERE condition` removes rows. Without `WHERE`, both act on **every** row.

In [16]:
demo.execute("INSERT INTO reviews (listing_id, month) VALUES (2, '2026-06-01')")    # n_reviews takes its DEFAULT
demo.execute("UPDATE listings SET price = 50 WHERE id = 2")                         # change one row
demo.execute("DELETE FROM reviews WHERE month < '2026-05-01'")                       # remove April
pd.read_sql_query("SELECT * FROM reviews", demo)

,listing_id,month,n_reviews
0,1,2026-05-01,2
1,2,2026-05-01,1
2,2,2026-06-01,1


### Loading many rows: executemany and commit

A loading script sends many rows at once. `executemany(sql, rows)` runs one statement for every tuple in a list. The `?` are **placeholders**: SQLite fills them with the values of each tuple (block 2 explains why values should always be passed this way). `commit()` ends the **transaction**: until then, the new rows are visible only to this connection and would be lost if the program stopped.

In [17]:
new_rows = [(4, "Pankow", "Private room", 60.0), (5, "Mitte", "Shared room", None)]
demo.executemany("INSERT INTO listings VALUES (?, ?, ?, ?)", new_rows)   # the ? are filled from each tuple
demo.commit()                                                            # make the changes permanent
print(demo.execute("SELECT COUNT(*) FROM listings").fetchone())

(5,)


### COPY, and what SQLite does instead

The curriculum names `COPY`. It is a **PostgreSQL** command that bulk-loads a file into a table in one step, much faster than one `INSERT` per row:

```sql
-- PostgreSQL, not SQLite
COPY listings (id, district, room_type, price) FROM '/data/listings.csv' WITH (FORMAT csv, HEADER true);
```

**SQLite has no `COPY` statement.** It offers three ways to do the same job:

| Way | Where it runs | Example |
|---|---|---|
| `.import` | the `sqlite3` command-line program (a dot command, not SQL) | `.import --csv --skip 1 listings.csv listings` |
| `executemany` | Python, standard library | read the file with `csv`, then `con.executemany("INSERT ...", rows)` |
| `DataFrame.to_sql` | Python with pandas | `df.to_sql("listings", con, if_exists="append", index=False)` |

The idea is the same in all of them (and in `COPY`): create the table with its rules first, load all rows in one transaction, then compare the row count with the source. The next cells try both Python ways. First we write 200 listings to a CSV file in a temporary folder (never into the repository), as if it came from a colleague.

In [18]:
import csv
import tempfile

tmp = Path(tempfile.mkdtemp())
sample = q("SELECT id, district, room_type, price FROM listings ORDER BY id LIMIT 200")
csv_path = tmp / "listings_sample.csv"
sample.to_csv(csv_path, index=False)
print(csv_path.name, csv_path.stat().st_size, "bytes")

listings_sample.csv 8165 bytes


#### Loading a CSV file with `executemany` and the `csv` module

`csv.reader` yields one list of strings per line; `next(reader)` skips the header. An empty field becomes `None`, which SQLite stores as NULL. The table was created with its rules first, so every row is checked.

In [19]:
load = sqlite3.connect(":memory:")
load.execute("""CREATE TABLE listings (id INTEGER PRIMARY KEY, district TEXT NOT NULL,
                                       room_type TEXT NOT NULL, price REAL CHECK (price > 0))""")
with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)                                                         # skip the header line
    rows = [[value if value != "" else None for value in row] for row in reader]
with load:                                                               # one transaction: commit at the end
    load.executemany("INSERT INTO listings VALUES (?, ?, ?, ?)", rows)
print("rows in file:", len(rows), "| rows in table:", load.execute("SELECT COUNT(*) FROM listings").fetchone()[0])

rows in file: 200 | rows in table: 200


#### Loading a DataFrame with `to_sql`

`DataFrame.to_sql(name, con, if_exists="append", index=False)` inserts all rows of a DataFrame into an existing table. Loading the same rows a second time breaks the primary key, and the whole batch is refused:

In [20]:
try:
    sample.to_sql("listings", load, if_exists="append", index=False)
except pd.errors.DatabaseError as error:          # pandas wraps the sqlite3.IntegrityError
    print("second load refused:", error.__cause__)    # the original SQLite message
print("rows in table:", load.execute("SELECT COUNT(*) FROM listings").fetchone()[0])

second load refused: UNIQUE constraint failed: listings.id
rows in table: 200


> **Note.** `DataFrame.to_sql(..., if_exists="replace")` drops the table and recreates it **without** primary keys or constraints. To keep them, create the table with `CREATE TABLE` first and load with `if_exists="append"`, as [`make_airbnb_sqlite.py`](make_airbnb_sqlite.py) does for `listings`.

### Schema first: what strict rules find in real data

The file [`sql/01-schema.sql`](sql/01-schema.sql) defines a strict version of the course database: every column has a type, keys and `CHECK` rules (coordinates inside Berlin, a known room type, real dates, `minimum_nights <= maximum_nights`). A date check uses `CHECK (date IS date(date))`: `date(x)` returns NULL for text that is not a valid date, and `IS` compares like `=` but treats NULL as an ordinary value, so `'2009-13-01'` fails. (`CHECK (date = date(date))` would let it pass, because a `CHECK` only rejects a row when its condition is *false*, and a comparison with NULL is *unknown*.)

Loading the real listings into this schema, exactly as they are in the file:

In [21]:
LISTING_COLS = ["id", "host_id", "district", "neighbourhood", "latitude", "longitude", "room_type", "accommodates",
                "bedrooms", "price", "minimum_nights", "maximum_nights", "availability_365", "number_of_reviews",
                "first_review", "last_review", "review_scores_rating", "license_status"]
strict = sqlite3.connect(":memory:")
strict.execute("PRAGMA foreign_keys = ON")
strict.executescript(Path("sessions/03-sql/workbooks/sql/01-schema.sql").read_text())
raw = q(f"SELECT {', '.join(LISTING_COLS)} FROM listings")
try:
    with strict:
        raw.to_sql("listings", strict, if_exists="append", index=False)
except pd.errors.DatabaseError as error:
    print("rejected:", error.__cause__)
print("listings stored:", strict.execute("SELECT COUNT(*) FROM listings").fetchone()[0])

rejected: CHECK constraint failed: maximum_nights < 2147483647
listings stored: 0


The database refuses the whole load, and because it was one transaction, not a single listing was stored. Which rows break the rule?

In [22]:
q("SELECT id, minimum_nights, maximum_nights FROM listings WHERE maximum_nights >= 2147483647")

,id,minimum_nights,maximum_nights
0,28004334,1,2147483647
1,50161791,1,2147483647


Two listings carry the maximum stay 2,147,483,647, the largest 32-bit integer: a software default, not a stay. The loader must now **decide** (set it to NULL, or drop the listings), and write the decision down. Extra 02 makes these decisions for all three tables and checks the counts against the source files; it is the model for the team project's loader.

*This subsection is a short version of [extras/02-loading-with-constraints.ipynb](extras/02-loading-with-constraints.ipynb) (course team, CC BY 4.0).*

### Loading data in practice

- **Data engineering teams** keep the `CREATE TABLE` statements of a project in version-controlled `.sql` files and review changes in pull requests, like any other code.
- **Public Health England** lost about 16,000 positive COVID-19 test results in October 2020 because a manual step used an old Excel file format whose row limit was exceeded (BBC News, 5 October 2020); a loading script with row-count checks would have caught it.

> 📖 **Read more:** [extras/02-loading-with-constraints.ipynb](extras/02-loading-with-constraints.ipynb) (course team): a strict course database with keys and checks for all three tables, the rows the rules reject, documented loading decisions and counts against the source files.

In [23]:
for c in (mem, demo, load, strict):     # close the temporary in-memory databases of this section
    c.close()

## Queries: SELECT, WHERE, ORDER BY, LIMIT

### The basic clauses

A query describes the result you want, not the steps to compute it; the database chooses how. SQL is therefore called a **declarative** language, unlike Python, where you write the steps. The basic clauses are:

- `SELECT` names the columns (or expressions) of the result; `*` means all columns; `AS` gives a result column a new name (an **alias**);
- `FROM` names the table;
- `WHERE` gives a condition that each row must satisfy;
- `ORDER BY` sorts the result (`ASC` ascending is the default, `DESC` descending);
- `LIMIT n` keeps only the first *n* rows; `OFFSET k` skips the first *k* rows before that.

Every query needs `SELECT` and (to read a table) `FROM`, and the clauses must be written in this order:

```
SELECT <column list>
FROM <table>
[WHERE <condition>]
[ORDER BY <column list>]
[LIMIT <number of rows>]
[OFFSET <number of rows>];
```

Square brackets mark optional clauses. By convention SQL keywords are written in capitals and each clause on its own line; SQL ignores upper and lower case in keywords and ignores line breaks, so both are only for the reader. A semicolon ends a statement; `read_sql_query` accepts a query with or without it.

*The clause template and conventions adapted from [Data 100, SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### The logical order of the clauses

Although we write `SELECT` first, the database evaluates the clauses in a different order:

```mermaid
flowchart LR
    F["1 FROM"] --> W["2 WHERE"] --> G["3 GROUP BY"] --> H["4 HAVING"] --> S["5 SELECT"] --> O["6 ORDER BY"] --> L["7 LIMIT / OFFSET"]
```

`GROUP BY` and `HAVING` follow in block 2. The order explains, for example, why `ORDER BY` can use an alias defined with `AS` in `SELECT`: it runs afterwards.

By hand: in the small tables from the start of the notebook, `SELECT id FROM listings WHERE district = 'Mitte' ORDER BY id` keeps listings 1 and 3 and returns them in this order.

### Why filtering in the database matters

Filtering in the database moves only the rows you need into Python. With 3.8 million calendar rows, `SELECT *` followed by filtering in pandas wastes memory and time. Writing the condition out also makes the question precise: "affordable short stays for a family" becomes `room_type = 'Entire home/apt' AND minimum_nights < 28 AND accommodates >= 4 AND price <= 150`.

### SELECT: columns, aliases, expressions and DISTINCT

Columns appear in the order you name them. An expression such as `price / accommodates` computes a new column for every row; `ROUND(x, 1)` rounds to one decimal, and `AS` names the result.

In [24]:
q("""
SELECT district, room_type, accommodates AS guests, price,
       ROUND(price / accommodates, 1) AS price_per_guest
FROM listings
LIMIT 5
""")

,district,room_type,guests,price,price_per_guest
0,Pankow,Entire home/apt,2,160.71,80.4
1,Pankow,Entire home/apt,7,193.33,27.6
2,Pankow,Entire home/apt,1,NaN,NaN
3,Friedrichshain-Kreuzberg,Entire home/apt,2,10.88,5.4
4,Pankow,Entire home/apt,4,372.67,93.2


`SELECT DISTINCT` removes duplicate rows from the result. With two columns, it keeps each different *combination* once.

In [25]:
print(q("SELECT DISTINCT room_type FROM listings"))
q("SELECT COUNT(*) AS n_combinations FROM (SELECT DISTINCT district, room_type FROM listings)")

         room_type
0  Entire home/apt
1     Private room
2      Shared room
3       Hotel room


,n_combinations
0,38


### WHERE: conditions

`WHERE` keeps the rows for which the condition is true. Conditions use the comparison operators `=`, `<>` (or `!=`), `<`, `<=`, `>`, `>=` and combine with `AND`, `OR` and `NOT`. Text values are written in single quotes: `'Mitte'`. (SQLite also accepts double quotes for text in some places, but double quotes are meant for column names; use single quotes.)

The best-rated entire flats in Neukölln under €100 a night that can be booked for less than 28 nights and have at least 50 reviews:

In [26]:
q("""
SELECT id, neighbourhood, price, number_of_reviews, review_scores_rating
FROM listings
WHERE district = 'Neukölln' AND room_type = 'Entire home/apt'
  AND minimum_nights < 28 AND price < 100 AND number_of_reviews >= 50    -- all conditions must hold
ORDER BY review_scores_rating DESC, number_of_reviews DESC             -- best rated first; reviews break ties
LIMIT 3
""")

,id,neighbourhood,price,number_of_reviews,review_scores_rating
0,31382644,Schillerpromenade,92.5,374,4.91
1,237670,Buckow Nord,68.5,91,4.84
2,190448,Buckow Nord,85.0,87,4.77


#### `AND`, `OR` and parentheses

**Predict.** `AND` binds more strongly than `OR`, as multiplication binds more strongly than addition. The two counts below differ only in the parentheses. Which is larger, and which one answers "private rooms in Mitte or Pankow"?

In [27]:
q("""
SELECT
  (SELECT COUNT(*) FROM listings
   WHERE room_type = 'Private room' AND district = 'Mitte' OR district = 'Pankow')   AS without_parentheses,
  (SELECT COUNT(*) FROM listings
   WHERE room_type = 'Private room' AND (district = 'Mitte' OR district = 'Pankow')) AS with_parentheses
""")

,without_parentheses,with_parentheses
0,2144,911


Without parentheses, the condition reads "(private room in Mitte) or (anything in Pankow)". Write the parentheses whenever `AND` and `OR` meet.

#### Shorter conditions: `IN`, `BETWEEN` and `NOT`

`IN (...)` tests membership in a list (like Python's `in`), `BETWEEN a AND b` tests a range with both ends included, and `NOT` negates either.

In [28]:
q("""
SELECT district, room_type, price
FROM listings
WHERE district IN ('Mitte', 'Pankow') AND room_type NOT IN ('Entire home/apt')
  AND price BETWEEN 50 AND 60              -- same as price >= 50 AND price <= 60
ORDER BY price
LIMIT 5
""")

,district,room_type,price
0,Pankow,Private room,50.00
1,Mitte,Private room,50.00
2,Pankow,Private room,50.84
3,Pankow,Hotel room,51.00
4,Pankow,Private room,51.30


#### Text patterns with `LIKE`

In a pattern, `%` stands for any number of characters (also none) and `_` for exactly one character. In SQLite, `LIKE` ignores upper and lower case for the letters A to Z (PostgreSQL's `LIKE` does not; it has `ILIKE` for that). How many listing titles mention a balcony, in English or German?

In [29]:
q("""
SELECT COUNT(*) AS n
FROM listings
WHERE name LIKE '%balcony%' OR name LIKE '%balkon%'
""")

,n
0,583


`_` matches exactly one character. Some neighbourhoods in Reinickendorf are called "West 1" to "West 5", and Charlottenburg has a "Westend". **Predict.** Which of them does `'West _'` match?

In [30]:
q("SELECT DISTINCT neighbourhood FROM listings WHERE neighbourhood LIKE 'West _' ORDER BY neighbourhood")

,neighbourhood
0,West 1
1,West 2
2,West 3
3,West 4
4,West 5


*`LIKE` with `%` and `_` adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause), here on the Airbnb data.*

#### Conditions on dates

Because dates are stored as `'YYYY-MM-DD'`, a date range is a text range: `date BETWEEN '2026-08-01' AND '2026-08-31'`. `strftime('%w', date)` returns the weekday as text (`'0'` is Sunday, `'6'` Saturday), and `strftime('%m', date)` the month. The share of Saturday nights in August that are still available in the calendar:

In [31]:
q("""
SELECT COUNT(*) AS saturday_nights, ROUND(AVG(available), 3) AS share_available
FROM calendar
WHERE date BETWEEN '2026-08-01' AND '2026-08-31' AND strftime('%w', date) = '6'
""")

,saturday_nights,share_available
0,51500,0.389


`AVG(available)` is the share of 1s, because `available` is 1 or 0. A night that is not available is booked or blocked by the host; the calendar does not say which.

### ORDER BY, LIMIT and OFFSET

`ORDER BY a, b` sorts by `a` and uses `b` to break ties; each column can have its own `DESC`. `LIMIT 3 OFFSET 3` returns rows 4 to 6 of the sorted result, which is how a website shows "page 2". The longest minimum stays:

In [32]:
q("""
SELECT id, district, room_type, minimum_nights
FROM listings
ORDER BY minimum_nights DESC, id
LIMIT 3 OFFSET 0       -- try OFFSET 3 for the next page
""")

,id,district,room_type,minimum_nights
0,6670861,Neukölln,Entire home/apt,1125
1,584757,Pankow,Private room,1000
2,6704144,Pankow,Private room,1000


A minimum stay of 1,125 nights, three years, is not a holiday rental; Session 4 treats such values.

> **Note.** Without `ORDER BY`, the order of rows is not defined. `LIMIT 10` without `ORDER BY` returns *some* ten rows, which may change between runs or database versions.

### CASE: a value that depends on a condition

`CASE WHEN condition THEN value ... ELSE value END` computes a value row by row, like `if`/`elif`/`else` in Python. The first true condition wins. Here each listing is put into a stay category; block 2 counts listings per category with `GROUP BY`.

In [33]:
q("""
SELECT id, minimum_nights,
       CASE WHEN minimum_nights < 7  THEN 'short stay'
            WHEN minimum_nights < 28 THEN 'week or more'
            ELSE 'month or more'
       END AS stay_type
FROM listings
ORDER BY id
LIMIT 5
""")

,id,minimum_nights,stay_type
0,3176,2,short stay
1,9991,6,short stay
2,14325,180,month or more
3,17904,93,month or more
4,20858,3,short stay


*`CASE` adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### Queries in practice

- **Property portals** turn the search form (district, number of rooms, maximum rent) into exactly such filter conditions before they sort the result by date or price, and use `LIMIT`/`OFFSET` for the result pages.
- **Data engineers** check every nightly load with small queries such as "how many orders arrived yesterday?".

> 📖 **Read more:** [extras/01-data100-sql-basics.ipynb](extras/01-data100-sql-basics.ipynb) (UC Berkeley Data 100, BSD 3-Clause): `SELECT`, `WHERE`, `ORDER BY`, `LIMIT` and `OFFSET` step by step on three-row toy tables, with the pandas equivalents.

## NULL values

### NULL and three-valued logic

**NULL** means "missing" or "unknown". It is not zero and not the empty text `''`. SQL treats it with **three-valued logic**: a condition is true, false or unknown.

- Any comparison with NULL is unknown: `price > 100` is unknown for a listing without a price, and so is `price = NULL`, and even `NULL = NULL`. `WHERE` keeps only rows whose condition is *true*, so rows with unknown conditions disappear from both `WHERE price > 100` and `WHERE NOT (price > 100)`.
- To test for NULL, write `price IS NULL` or `price IS NOT NULL`. In SQLite, `a IS b` also compares two values and treats two NULLs as equal (standard SQL: `IS NOT DISTINCT FROM`).
- Arithmetic and text operations with NULL give NULL: `price * 2` is NULL when the price is, and so is `'Berlin' || NULL`.
- `COALESCE(price, 0)` returns the first value that is not NULL, here 0 for missing prices. Use it only when "missing" really means zero.
- Counting: `COUNT(*)` counts rows; `COUNT(price)` counts only the rows where `price` is not NULL. Functions such as `AVG(price)` and `SUM(price)` skip NULLs.
- Sorting: SQLite sorts NULL before every other value (first in ascending order, last in descending order). PostgreSQL does the opposite. `ORDER BY price NULLS LAST` makes the order explicit in both.

| `a` | `b` | `a AND b` | `a OR b` | `NOT a` |
|---|---|---|---|---|
| true | unknown | unknown | true | false |
| false | unknown | false | unknown | true |
| unknown | unknown | unknown | unknown | unknown |

By hand, with the three small listings from the start of the notebook: `WHERE price < 100` keeps only listing 2; `WHERE price >= 100` keeps only listing 1; listing 3 is in neither result. `COUNT(*)` is 3, `COUNT(price)` is 2, and the average price is (120 + 45) / 2 = 82.5, not (120 + 45 + 0) / 3.

### Why NULL matters

A third of the listings in the case study show no price, so NULL changes almost every answer about prices. A filter that forgets NULL silently changes the population a number describes, and a count over a column instead of over rows silently answers a different question.

### NULL in conditions and counts

Every listing is above €150, not above €150, or without a price. **Predict.** Do the first two counts add up to all listings?

In [34]:
q("""
SELECT (SELECT COUNT(*) FROM listings WHERE price > 150)          AS above_150,
       (SELECT COUNT(*) FROM listings WHERE NOT (price > 150))    AS not_above_150,
       (SELECT COUNT(*) FROM listings WHERE price IS NULL)        AS no_price,
       (SELECT COUNT(*) FROM listings WHERE price = NULL)         AS equals_null,
       (SELECT COUNT(*) FROM listings)                            AS all_listings
""")

,above_150,not_above_150,no_price,equals_null,all_listings
0,2846,3908,3467,0,10221


Only the first three numbers together give all listings: the listings without a price fall out of both the condition and its negation, and only `IS NULL` finds them. `price = NULL` finds nothing, because the comparison is never true.

### NULL in expressions, `COALESCE` and the sort order

In ascending order SQLite puts the listings without a price first; `NULLS LAST` moves them to the end.

In [35]:
q("""
SELECT id, room_type, price, price * 2 AS double_price, COALESCE(price, -1) AS price_or_minus_one
FROM listings
WHERE district = 'Mitte'
ORDER BY price          -- ascending: SQLite puts the NULLs first; try ORDER BY price NULLS LAST
LIMIT 3
""")

,id,room_type,price,double_price,price_or_minus_one
0,191023,Private room,None,None,-1
1,251406,Entire home/apt,None,None,-1
2,499342,Entire home/apt,None,None,-1


### Is a missing price random?

A listing whose calendar has no available night in the coming year (`availability_365 = 0`) cannot show the price of a next free night.

In [36]:
q("""
SELECT (SELECT COUNT(*) FROM listings WHERE price IS NULL)                              AS no_price,
       (SELECT COUNT(*) FROM listings WHERE price IS NULL AND availability_365 = 0)     AS no_price_no_free_night,
       (SELECT COUNT(*) FROM listings WHERE price IS NOT NULL AND availability_365 = 0) AS price_no_free_night
""")

,no_price,no_price_no_free_night,price_no_free_night
0,3467,2744,1


Four in five listings without a price have no free night at all: they are probably not offered at the moment. Any statement about "the price of a listing in Berlin" therefore describes the listings that can be booked, not all listings. Session 4 returns to such missing values.

### NULL in practice

- **Surveys** distinguish "answered 0" from "did not answer"; storing both as 0 makes the average wrong.
- **Health records** leave a laboratory value empty when the test was not done; treating empty as "normal" or as zero leads to wrong conclusions.

> **Note.** `WHERE x NOT IN (...)` behaves unexpectedly when the list contains a NULL. Block 2 shows safer ways to find rows without a partner.

## Worked example · Which flat can a family book for a weekend?

A family of four looks for an entire flat in Friedrichshain-Kreuzberg for a weekend: a minimum stay of at most two nights, at least four guests, at most €150 a night, and at least 20 reviews. ### Step 1 · Turn the family's wishes into conditions and a sort order

Each part of the question becomes one condition; the best-rated flats come first, and the number of reviews breaks ties.

**Why this step?** Why sort by rating and then by number of reviews, and not by rating alone?

In [37]:
family = q("""
SELECT id, neighbourhood, accommodates, price, number_of_reviews, review_scores_rating
FROM listings
WHERE district = 'Friedrichshain-Kreuzberg'
  AND room_type = 'Entire home/apt'
  AND minimum_nights <= 2 AND accommodates >= 4 AND price <= 150 AND number_of_reviews >= 20
ORDER BY review_scores_rating DESC, number_of_reviews DESC
LIMIT 5
""")
family

,id,neighbourhood,accommodates,price,number_of_reviews,review_scores_rating
0,20660494,Tempelhofer Vorstadt,4,128.67,361,4.88
1,51830435,Frankfurter Allee Süd FK,6,128.00,271,4.86
2,48732044,Karl-Marx-Allee-Süd,6,129.00,298,4.85
3,15887444,Südliche Friedrichstadt,4,125.00,23,4.83
4,26633041,Tempelhofer Vorstadt,4,92.33,372,4.80


### Step 2 · Check the calendar of the best flat

Is the best-rated flat still free for the first weekend of August 2026? We take its id from the result and pass it to the calendar query with a `?` placeholder (block 2 explains why this is safer than pasting the id into the text).

**Why this step?** The flat passed every condition of step 1. Why look at a second table at all?

In [38]:
best = int(family.loc[0, "id"])
q("""
SELECT date, strftime('%w', date) AS weekday, available
FROM calendar
WHERE listing_id = ? AND date BETWEEN '2026-07-31' AND '2026-08-02'
ORDER BY date
""", params=(best,))

,date,weekday,available
0,2026-07-31,5,0
1,2026-08-01,6,0
2,2026-08-02,0,0


### Step 3 · Read the answer and decide

In this snapshot all three nights are unavailable, so the family moves on to the second flat of the list. The answer comes from two small queries on two tables, linked by the key `id` = `listing_id`. Block 2 does such links in one query with a join.

## Check your understanding

Vote alone first, discuss with your neighbour, then vote again. Each question has one best answer.

**1.** A database has foreign keys switched on (`PRAGMA foreign_keys = ON`), and `reviews.listing_id` refers to `listings.id`. What happens to `INSERT INTO reviews VALUES (9, '2026-05-01', 1)` when there is no listing 9?

- **A.** The row is stored, and a later join drops it.
- **B.** The row is stored with `listing_id` set to NULL.
- **C.** The insert fails with `FOREIGN KEY constraint failed`, and nothing is stored.
- **D.** SQLite adds an empty listing 9, so that the key holds.

**2.** SQLite has no date type. Why does `ORDER BY first_review` still put the dates in the right order?

- **A.** SQLite recognises dates and converts them to day numbers before sorting.
- **B.** It does not: dates must be converted with `julianday()` before they can be sorted.
- **C.** SQLite stores the dates as integers such as 20250105.
- **D.** Text of the form `'YYYY-MM-DD'` is compared character by character, and the year comes first, then the month, then the day, each with a fixed number of digits.

**3.** In which order does the database evaluate the clauses of `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT`?

- **A.** `FROM`, `WHERE`, `SELECT`, `ORDER BY`, `LIMIT`
- **B.** `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT`
- **C.** `FROM`, `SELECT`, `WHERE`, `LIMIT`, `ORDER BY`
- **D.** In the order they are written; the optimiser only makes it faster.

**4.** 3,467 listings have no price, yet `SELECT COUNT(*) FROM listings WHERE price = NULL` returns 0. Why?

- **A.** The missing prices are stored as 0, so they never equal NULL.
- **B.** `COUNT(*)` skips the rows in which a column is NULL.
- **C.** A comparison with NULL is unknown for every row, and `WHERE` keeps only the rows whose condition is true.
- **D.** NULL is stored as the text `'NULL'`, so the comparison needs quotes.

**5.** SQLite has no `COPY`. Which way loads a CSV file into a table so that the primary key and the other constraints still hold?

- **A.** `df.to_sql("listings", con, if_exists="replace")`, which also recreates the table.
- **B.** `CREATE TABLE` with the key and the rules first, then `df.to_sql(..., if_exists="append")` or `executemany` in one transaction, then compare the row count with the file.
- **C.** `con.execute(".import --csv listings.csv listings")` from Python.
- **D.** `executemany` with one `commit()` per row, so that a failing row stops the load.

## Exercises

Work in pairs; the lecturer picks which exercises to set. The cells above have opened the course database as `con` and defined `q(sql, params)`, which returns a query result as a DataFrame. For **Paper** and **Predict** tasks, write your answer in the markdown cell before you run anything. *Stretch* tasks are for pairs that finish early.

### 1.1 · Warm-ups: predict the output

*Predict*

> What does the database return?
> 1. `SELECT 1/2;`
> 2. `SELECT NULL = NULL;`
> 3. `SELECT 'Berlin' || NULL;`
> 4. `SELECT COUNT(*), COUNT(price), AVG(price) FROM listings;` (a third of the prices are missing)
>
> Write your four predictions first, then check them with `q(...)`.

*Source:* [SQLite: Quirks, caveats and gotchas](https://www.sqlite.org/quirks.html); PostgreSQL docs: [mathematical operators](https://www.postgresql.org/docs/current/functions-math.html), [comparison operators](https://www.postgresql.org/docs/current/functions-comparison.html)

*Your answer:*

### 1.2 · Fermi estimate

*Paper*

> Without looking at the data: how many rows of the review table (one row per listing and month with at least one review) belong to the 2,268 listings in Mitte? Write down each assumption of your estimate (how many listings have any review, for how many months on average). Then check your estimate with one query.

*Source:* [Fermi problem, Wikipedia](https://en.wikipedia.org/wiki/Fermi_problem)

*Your answer:*

### 1.3 · Count the rows of each join

*Paper*

> Table `a` has the rows (1, x), (2, y), (2, z); table `b` has (2, p), (2, q), (3, r). The first column is the key. How many rows do `a INNER JOIN b`, `a LEFT JOIN b` and `a CROSS JOIN b` return?

*Source:* original exercise for this course; background: [PostgreSQL: Joined tables](https://www.postgresql.org/docs/current/queries-table-expressions.html)

*Your answer:*

### 1.4 · Why can't `WHERE` use a `SELECT` alias?

*Paper*

> This query fails in PostgreSQL. Why?
> ```sql
> SELECT id, price / accommodates AS price_per_guest
> FROM listings
> WHERE price_per_guest > 50;
> ```
> Then run it on the SQLite file. What do you notice?

*Source:* [PostgreSQL: SELECT, general processing](https://www.postgresql.org/docs/current/sql-select.html)

*Your answer:*

### 1.5 · First queries on the listings

*Code*

> Answer each question with one query.
> 1. How many listings require a minimum stay of 28 nights or more?
> 2. A student looks for a cheap room in Pankow: a private room under €50 a night, a minimum stay below 28 nights, and at least 10 reviews. List these listings with id, neighbourhood, price, rating and number of reviews, best rated first. Does any price look suspicious?
> 3. How many listings in Mitte have never been reviewed according to `first_review IS NULL`? Is that the same number as with `number_of_reviews = 0`?

*Source:* original exercise for this course (Airbnb case study)

### 1.6 · Create and fill a table

*Code*

> In a new in-memory database (`sqlite3.connect(":memory:")`), create a table `trips` for your own last journeys with the columns `id` (primary key), `city` (text, required), `start_date` (text `'YYYY-MM-DD'`), `nights` (integer, must be at least 1) and `price_per_night` (decimal, may be unknown). Insert four trips, one of them with an unknown price. Then try to insert a trip with an `id` that exists, one with 0 nights and one without a city, and read the error messages. Finally, list the trips longer than three nights, the most recent first. What is the total cost of all trips, and how does the trip with an unknown price affect it?

*Source:* original exercise for this course; background: [SQLite: CREATE TABLE](https://www.sqlite.org/lang_createtable.html)

### 1.7 · Airbnb detective

*Pairs*

> Follow the clues; each answer is needed for the next query. Submit the final answer on paper.
> 1. Which host has the most listings in Berlin?
> 2. In which district does that host have the most listings?
> 3. In that district, which listing (of any host) has the most reviews?
> 4. In which month did that listing receive the most reviews?
>
> *Note:* clues 1 and 2 count rows per host or per district. This needs `COUNT(*)` with `GROUP BY`, which block 2 teaches in full; the pattern is `SELECT host_id, COUNT(*) AS n FROM listings GROUP BY host_id ORDER BY n DESC`.

*Source:* original exercise for this course, modelled on the [SQL Murder Mystery, Knight Lab](https://github.com/NUKnightLab/sql-mysteries)

### 1.8 · How does the database find a row? (index and query plan)

*Stretch:* *Predict, then Code*

> We look up the calendar of one listing: `SELECT * FROM calendar WHERE listing_id = 3176;`
> Predict: how many of the 3.8 million rows does the database read? Then run `EXPLAIN QUERY PLAN` before and after `CREATE INDEX idx_calendar_listing ON calendar(listing_id);` and time both queries.
>
> When you are done, remove the index again with `DROP INDEX idx_calendar_listing;` so that the file is as before. (Hint for timing: `import time` and `time.perf_counter()`.)

*Source:* [SQLite: EXPLAIN QUERY PLAN](https://www.sqlite.org/eqp.html)

*Your answer:*

### 1.9 · Put the query in order

*Parsons*

> The lines below form a query that lists the five cheapest entire homes in Mitte for at least four guests that can be booked for less than 28 nights, cheapest first. Put them in the right order. Then say which line changes the result most if you leave it out, and why.
>
> ```sql
> ORDER BY price
> AND accommodates >= 4 AND minimum_nights < 28
> SELECT id, price, accommodates
> LIMIT 5
> WHERE district = 'Mitte'
> AND price IS NOT NULL
> FROM listings
> AND room_type = 'Entire home/apt'
> ```

*Source:* original exercise for this course

*Your answer:*

### 1.10 · Listings first reviewed since January 2025

*Find the bug*

> A colleague counts the listings whose first review is on or after 5 January 2025:
>
> ```sql
> SELECT COUNT(*) FROM listings WHERE first_review >= '2025-1-5'
> ```
>
> The query runs and returns a plausible number, but the number is wrong. What does the condition actually compare, which first reviews does it count and which does it miss? Fix the query and compare the two counts.

*Source:* original exercise for this course; background: [SQLite: Date and time functions](https://www.sqlite.org/lang_datefunc.html)

*Your answer:*

## Debrief

- **Joins on paper (1.3).** Collect the three numbers from several pairs. An answer of 2 rows for the inner join is the typical mistake: the duplicated key on both sides multiplies rows. Ask what this means for a join of `listings` with 365 calendar rows each.
- **Warm-ups (1.1).** Ask why `NULL = NULL` is not true, and what `COUNT(price)` counts compared with `COUNT(*)`. Point to integer division in `1/2`: it returns a surprise again in block 2 when shares are computed.
- **Aliases (1.4).** Compare the logical order of the clauses with what SQLite accepts: SQLite is lenient, PostgreSQL is not. Test SQL on the database you will deploy to.
- **First queries (1.5).** Compare the two counts of never-reviewed listings in Mitte and ask which column is the better source and why.
- **Bridge to block 2.** The detective needed counts per host and per district (`GROUP BY`), and the worked example needed two queries for two tables. Block 2 adds aggregation, joins, common table expressions, window functions, and runs SQL from Python scripts.

## Further reading

- Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM*, 13(6), 377–387. https://doi.org/10.1145/362384.362685
- UC Berkeley Data 100, course notes Fall 2023: [SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) and [SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (BSD 3-Clause). Curated into the sections on relational databases, keys, creating tables and queries; kept as [extras/01-data100-sql-basics.ipynb](extras/01-data100-sql-basics.ipynb) and [extras/03-data100-sql-having-cleaning-joins.ipynb](extras/03-data100-sql-having-cleaning-joins.ipynb).
- Course team: [extras/02-loading-with-constraints.ipynb](extras/02-loading-with-constraints.ipynb), schema-first loading of the whole case study.
- SQLite documentation: [Datatypes](https://www.sqlite.org/datatype3.html), [STRICT tables](https://www.sqlite.org/stricttables.html), [CREATE TABLE](https://www.sqlite.org/lang_createtable.html), [SELECT](https://www.sqlite.org/lang_select.html), [Quirks, caveats and gotchas](https://www.sqlite.org/quirks.html), [Command-line shell (`.import`)](https://www.sqlite.org/cli.html), [Most widely deployed database engine](https://www.sqlite.org/mostdeployed.html).
- PostgreSQL documentation: [COPY](https://www.postgresql.org/docs/current/sql-copy.html).
- Library of Congress. *Recommended formats statement: datasets*. https://www.loc.gov/preservation/resources/rfs/data.html
- Software Carpentry. *Databases and SQL* (lessons with SQLite, CC BY 4.0). https://swcarpentry.github.io/sql-novice-survey/
- [SQLBolt](https://sqlbolt.com/), interactive lessons 1–6 (queries, filtering, sorting), and the [SQL Murder Mystery](https://mystery.knightlab.com/) of Knight Lab.